# Ploemeur temporal example

This notebook is the guided entry point for the temporal `ploemeur` example.

Default behavior:
- inspect the multi-date observation record,
- run the temporal MH workflow,
- read one observation overview and one compact figure set per LPM.

Additional diagnostics are available in an optional expert section.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for parent in [ROOT, *ROOT.parents]:
    if (parent / 'pyproject.toml').exists() and (parent / 'pyage').exists():
        ROOT = parent
        break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import Image, Markdown, display
import pandas as pd
import yaml
import pyage.concentrations.concentrations as co
from scripts.common.example_summary_plots import plot_observations_overview
from scripts.launcher_temporal import run_temporal

def read_tsv(path):
    frame = pd.read_csv(path, sep='	')
    return frame.loc[:, ~frame.columns.str.startswith('Unnamed')]

def read_stats(path):
    return pd.read_csv(path, sep='	', index_col=0)

print('ROOT:', ROOT)

## Notebook mode

Keep `expert_mode = False` for a first pass.
You can also restrict the display to one LPM with `focus_lpm`.

In [ ]:
expert_mode = False
focus_lpm = None  # Example: 'exp_shifted'

params_path = ROOT / 'examples' / 'ploemeur_temporal' / 'ploemeur_temporal.yaml'
params = yaml.safe_load(params_path.read_text(encoding='utf-8'))

display_lpms = params['lpm_models']['list']
if focus_lpm is not None:
    display_lpms = [focus_lpm]

print('Config:', params_path)
print('Dataset:', params['dataset']['file'])
print('Mode:', params['workflow']['mode'])
print('Displayed LPMs:', display_lpms)
print('Metropolis-Hastings steps:', params['calibration']['mh_nsteps'])
print('Expert mode:', expert_mode)

## Step 1 - Inspect the observed time series

Start with the data only.
The observation overview is the simplest way to understand the temporal coverage, the tracer spread, and the outliers before looking at any model fit.

In [ ]:
dataset_path = ROOT / params['dataset']['file']
cdata = co.Concentrations(file_load=True, file_name=str(dataset_path))
error_rel = params['dataset'].get('error_rel')
if error_rel is not None and (cdata.cv['error'] == 0).any():
    cdata.error_affect_from_value(float(error_rel))

display(cdata.cv.head(12))
plot_observations_overview(cdata, title='Observed concentrations before calibration')

## Step 2 - Run the temporal workflow

The launcher writes one observation overview and, for each LPM, a temporal fit summary plus a compact parameter summary.

In [ ]:
results_root = run_temporal(params_path)
results_root = Path(results_root)
print('Results root:', results_root)

## Step 3 - Beginner view: one compact figure set per LPM

Read the outputs in this order:
1. observation overview,
2. temporal fit summary,
3. parameter summary.

In [ ]:
overview = results_root / '00_observations_overview.png'
if overview.exists():
    display(Markdown('### Observation overview'))
    display(Image(filename=str(overview)))
    display(Markdown("Observation overview: start here before reading any model output.\n\nThis figure shows the raw temporal spread, the uncertainty bars, and the main outliers for each tracer. It tells you what the model will have to explain."))

for lpm_name in display_lpms:
    lpm_dir = results_root / lpm_name
    fit_path = lpm_dir / 'Metropolis_Hastings' / 'concentration_times.png'
    param_path = lpm_dir / 'parameter_summary.png'
    display(Markdown(f'## {lpm_name}'))
    if fit_path.exists():
        display(Markdown('### Temporal fit summary'))
        display(Image(filename=str(fit_path)))
        display(Markdown("Temporal fit summary: the dark line is the median calibrated response and the shaded bands summarize posterior uncertainty.\n\nRead this figure by checking whether the observations sit inside the uncertainty bands and whether the model misses specific years or tracer ranges."))
    if param_path.exists():
        display(Markdown('### Parameter summary'))
        display(Image(filename=str(param_path)))
        display(Markdown("Parameter summary: each panel shows the posterior spread of one calibrated parameter for this LPM.\n\nUse it to judge whether the temporal dataset strongly constrains the parameter or leaves a broad range of plausible values."))

## Step 4 - Read the calibrated parameter tables

After the figures, the next useful object is the per-LPM statistics table.
Read each LPM independently before comparing them.

In [ ]:
for lpm_name in display_lpms:
    stats_path = results_root / lpm_name / 'lpm_stats_calibrated.txt'
    stats = read_stats(stats_path)
    display(Markdown(f'### {lpm_name}'))
    display(stats.loc[['count', 'mean', 'std']])

## Expert mode (optional)

Set `expert_mode = True` only if you want to inspect the raw calibrated samples, the full file tree, or optional advanced diagnostics.

In [ ]:
if expert_mode:
    for lpm_name in display_lpms:
        lpm_dir = results_root / lpm_name
        display(Markdown(f'## {lpm_name} expert diagnostics'))
        dist_path = lpm_dir / 'lpm_dist_calibrated.txt'
        stats_path = lpm_dir / 'lpm_stats_calibrated.txt'
        if dist_path.exists():
            display(Markdown('### First calibrated samples'))
            display(read_tsv(dist_path).head(10))
            display(Markdown("These are raw posterior samples.\n\nUse them only when you need to inspect the calibrated distribution in detail or debug one parameter trend manually."))
        if stats_path.exists():
            display(Markdown('### Full statistics table'))
            display(read_stats(stats_path))
            display(Markdown("The full statistics table is useful when you want more than the mean and standard deviation, for example quantiles or the objective-function summary."))

        pair_plots = sorted(lpm_dir.glob('concentrations2D_*.png'))
        if pair_plots:
            display(Markdown('### Example 2D concentration diagnostic'))
            display(Image(filename=str(pair_plots[0])))
            display(Markdown("This kind of 2D concentration diagnostic is an expert view.\n\nIt is mainly useful for checking the internal geometry of the calibrated cloud, not for a first interpretation of the example."))

    display(Markdown('### Result folders'))
    for path in sorted(results_root.iterdir()):
        print(path.name)
else:
    print('Set expert_mode = True and re-run this cell to display advanced diagnostics.')